# Wild cluster bootstrap-$t$ inference for the Instagram DIDs

The two Instagram DIDs each have only 13–14 location clusters, with one location (Aïn Sebaâ) contributing ~63 % of all responses. Conventional cluster-robust (CR1) standard errors are unreliable here (Cameron, Gelbach & Miller 2008; MacKinnon & Webb 2017). The recommended fix is the **wild cluster bootstrap-$t$**, which simulates the null distribution of the test statistic under cluster-level resampling and is robust to small numbers of unequal-size clusters.

We use the **linear probability model** (weighted OLS, weights = total responses per ad) rather than the binomial GLM. The wild cluster bootstrap-$t$ is well defined in the linear case and coefficients are interpreted directly in percentage points. We run:

1. Communal-prayer DID (Friday + Sunday): $H_0: \beta_{\text{after}\times\text{own\_prayer\_day}} = 0$.
2. Individual-prayer DID (Tuesday/Dhuhr): $H_0: \beta_{\text{after}\times\text{is\_muslim}} = 0$.
3. Two falsification tests (Sunday × Muslim, Friday × Christian).

Using self-contained Rademacher-weighted wild cluster bootstrap with B = 1999. Restricted-residual variant (residuals come from a model that imposes $H_0$), the recommended choice with few clusters (MacKinnon & Webb 2017).

Data dependencies: Needs `midsave/df_well_glm.parquet` from `did_analysis.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from patsy import dmatrices
from pathlib import Path

In [ ]:
REPO = Path('..')
RNG  = np.random.default_rng(20260506)
B    = 1999  # bootstrap replications

In [ ]:
weather_covs = ['temperature_2m','precipitation','cloud_cover','wind_speed_10m','relative_humidity_2m']
demog_covs   = ['is_female','age_25_49','age_50_65']

## Load the analysis dataset

Built and saved by `did_analysis.ipynb`. We rebuild only the regressor columns we need.

In [ ]:
path = REPO / 'midsave/df_well_glm.parquet'
if not path.exists():
    raise FileNotFoundError(
        f'{path} not found.'
        '(the data-prep section now writes this file).'
    )
df = pd.read_parquet(path).copy()

df['total_responses'] = df['Yes_Well'] + df['No_Well']
df['prop_good']       = df['Yes_Well'] / df['total_responses']
df['after_int']       = df['after'].astype(int)
df['is_muslim']       = (df['Religion'] == 'muslim').astype(int)
df['is_female']       = (df['Gender']   == 'Female').astype(int)
df['age_25_49']       = (df['Age']      == '(25, 49)').astype(int)
df['age_50_65']       = (df['Age']      == '(50, 65)').astype(int)
df['own_prayer_day']  = (
    ((df['Religion']=='muslim')    & (df['day_of_week_start']=='Friday')) |
    ((df['Religion']=='christian') & (df['day_of_week_start']=='Sunday'))
).astype(int)

print(f'ads:               {len(df):>5}')
print(f'locations:         {df.Location.nunique():>5}')
print(f'total responses:   {df.total_responses.sum():>6,}')
print('\nresponses by day_of_week × Religion:')
print(df.groupby(['day_of_week_start','Religion'])['total_responses'].sum().to_string())

## Wild cluster bootstrap-$t$

For WLS with weights $w$ and design $X$:

1. Fit unrestricted: $\hat\beta = (X'WX)^+ X'Wy$. Get observed $t = \hat\beta_k / \widehat{\mathrm{SE}}_k^{\text{CR1}}$.
2. Fit restricted (impose $\beta_k = 0$): $\tilde\beta$ on $X_{-k}$, residuals $\tilde u = y - X_{-k}\tilde\beta$, fitted $\tilde y = X_{-k}\tilde\beta$.
3. For $b = 1, \dots, B$: draw Rademacher $w_g \in \{-1,+1\}$ per cluster; form $y^*_i = \tilde y_i + w_{g(i)}\,\tilde u_i$; refit unrestricted; compute CR1 $t^*_b$ on $\beta_k$.
4. Two-sided $p = (1 + \#\{|t^*_b| \ge |t|\})/(B+1)$.

Solve WLS via $\sqrt{w}$-rescaling and `np.linalg.lstsq`.

In [ ]:
def wls_fit(X, y, w):
    """WLS via sqrt(w)-rescaling. Returns coefficient vector."""
    sw = np.sqrt(w)
    beta, *_ = np.linalg.lstsq(X * sw[:,None], y * sw, rcond=None)
    return beta

def cr1_se(X, w, resid, cluster_idx, G, var_col):
    """Cluster-robust (CR1) SE for one coefficient.
    bread = (X' W X)^+;  meat = sum_g (X_g' W_g resid_g)(X_g' W_g resid_g)'.
    """
    n, k = X.shape
    bread = np.linalg.pinv((X * w[:,None]).T @ X)
    XtuG  = np.zeros((G, k))
    np.add.at(XtuG, cluster_idx, X * (w * resid)[:,None])
    V = bread @ (XtuG.T @ XtuG) @ bread
    correction = (G/(G-1)) * ((n-1)/(n-k))
    return np.sqrt(max(V[var_col, var_col] * correction, 0.0))

def wild_cluster_bootstrap(data, formula, var, cluster='Location',
                            weight_col='total_responses', B=1_999, rng=RNG):
    yvec, Xfull = dmatrices(formula, data=data, return_type='dataframe')
    if var not in Xfull.columns:
        raise KeyError(f'{var} not in design columns')
    Xres = Xfull.drop(columns=[var])

    y, Xf, Xr = yvec.values.ravel(), Xfull.values, Xres.values
    w = data[weight_col].astype(float).values

    groups      = data[cluster].values
    unique_g    = np.unique(groups)
    G           = len(unique_g)
    cluster_idx = pd.Categorical(groups, categories=unique_g).codes
    var_col     = list(Xfull.columns).index(var)

    beta_u   = wls_fit(Xf, y, w)
    coef_obs = beta_u[var_col]
    se_obs   = cr1_se(Xf, w, y - Xf @ beta_u, cluster_idx, G, var_col)
    t_obs    = coef_obs / se_obs

    beta_r = wls_fit(Xr, y, w)
    yhat_r = Xr @ beta_r
    u_r    = y - yhat_r

    t_star = np.empty(B)
    for b in range(B):
        wg     = rng.choice([-1.0, 1.0], size=G)
        y_b    = yhat_r + u_r * wg[cluster_idx]
        beta_b = wls_fit(Xf, y_b, w)
        se_b   = cr1_se(Xf, w, y_b - Xf @ beta_b, cluster_idx, G, var_col)
        t_star[b] = beta_b[var_col] / se_b if se_b > 0 else 0.0

    p_boot = (1 + np.sum(np.abs(t_star) >= abs(t_obs))) / (B + 1)
    q = np.quantile(t_star, [0.025, 0.975])
    return {
        'coef':         coef_obs,
        'se_cr1':       se_obs,
        't':            t_obs,
        'p_boot':       p_boot,
        'ci95_boot_lo': coef_obs - q[1] * se_obs,
        'ci95_boot_hi': coef_obs - q[0] * se_obs,
        'G':            G,
        'n':            len(y),
        'B':            B,
    }

## Naive CR1 reference

Conventional cluster-robust $p$-value via statsmodels.

In [ ]:
def naive_cr1(data, formula, var, cluster='Location', weight_col='total_responses'):
    weights = data[weight_col].astype(float)
    m = smf.wls(formula, data=data, weights=weights).fit(
        cov_type='cluster', cov_kwds={'groups': data[cluster]})
    return {'coef_sm': m.params[var], 'se_sm': m.bse[var], 'p_sm': m.pvalues[var]}

## Run the four tests

Preferred specifications (those reported as the paper's column 4): demographic + weather controls and location fixed effects.

In [ ]:
def prune(formula, data):
    """Drop covariates with zero within-sample variation."""
    out = formula
    for c in weather_covs + demog_covs:
        if c in data.columns and data[c].nunique() < 2:
            out = out.replace(f' + {c}','').replace(f'{c} + ','')
    return out

In [ ]:
demog_str   = ' + '.join(demog_covs)
weather_str = ' + '.join(weather_covs)

df_comm = df[df['day_of_week_start'].isin(['Friday','Sunday'])].copy()
df_tue  = df[df['day_of_week_start']=='Tuesday'].copy()
df_f1   = df[(df['day_of_week_start']=='Sunday') & (df['Religion']=='muslim')].copy()
df_f2   = df[(df['day_of_week_start']=='Friday') & (df['Religion']=='christian')].copy()

f_comm = f'prop_good ~ after_int * own_prayer_day + {demog_str} + {weather_str} + C(Location)'
f_ind  = f'prop_good ~ after_int * is_muslim     + {demog_str} + {weather_str} + C(Location)'
f_f1   = f'prop_good ~ after_int                 + {demog_str} + {weather_str}'
f_f2   = f'prop_good ~ after_int                 + {demog_str} + {weather_str}'

In [ ]:
tests = [
    ('Communal DID (Fri+Sun)',         df_comm, prune(f_comm, df_comm), 'after_int:own_prayer_day'),
    ('Individual DID (Tue/Dhuhr)',     df_tue,  prune(f_ind,  df_tue),  'after_int:is_muslim'),
    ('Falsification: Sun × Muslim',    df_f1,   prune(f_f1,   df_f1),   'after_int'),
    ('Falsification: Fri × Christian', df_f2,   prune(f_f2,   df_f2),   'after_int'),
]

In [ ]:
rows = []
for label, data, formula, var in tests:
    print(f'\n→ {label}  (n_ads={len(data)}, G={data.Location.nunique()})')
    sm_res = naive_cr1(data, formula, var)
    boot   = wild_cluster_bootstrap(data, formula, var, B=B)
    rows.append({'Test': label, **sm_res, **boot})
    print(f"   coef            = {boot['coef']:+.4f}")
    print(f"   CR1 SE          = {boot['se_cr1']:.4f}")
    print(f"   p (CR1, asymp.) = {sm_res['p_sm']:.4f}")
    print(f"   p (wild boot)   = {boot['p_boot']:.4f}    [B={B}]")
    print(f"   95% boot CI     = [{boot['ci95_boot_lo']:+.4f}, {boot['ci95_boot_hi']:+.4f}]")

## Summarize and Save

Coefficients are on the proportion scale (units of one in `prop_good`). Multiply by 100 for percentage points.

In [ ]:
summary = pd.DataFrame(rows)[
    ['Test','coef','se_cr1','p_sm','p_boot','ci95_boot_lo','ci95_boot_hi','G','n']
].rename(columns={'p_sm':'p_naive_cr1','p_boot':'p_wildboot'})
for c in ['coef','se_cr1','p_naive_cr1','p_wildboot','ci95_boot_lo','ci95_boot_hi']:
    summary[c] = summary[c].round(4)
summary

In [ ]:
out = REPO / 'output/wild_bootstrap_summary.csv'
summary.to_csv(out, index=False)

- A bootstrap $p$-value much **larger** than the naive CR1 $p$-value indicates that the asymptotic CR1 inference was anti-conservative — what we expect with $G \le 14$ unequal clusters.
- A bootstrap $p$-value **comparable** to the naive one indicates that CR1 was reasonably calibrated.
- For the falsification rows: a significant $\beta_{\text{after}}$ confirms the generic morning-to-afternoon decline. This is *not* a problem for the DID — it is differenced out by the interaction — but it should pass the bootstrap test in the same direction.